In [1]:
import json
import numpy as np
from pathlib import Path
from typing import Dict, List

from model_ranking import (
    load_h5,
    get_ckpt_eval_scores,
    find_selftraining_pred_paths,
    get_NA_prediction_path,
    MODEL_ABBREVIATIONS_TO_DATASET

)

from pytorch3dunet.unet3d.config import load_config_direct #pyright: ignore[reportUnknownVariableType]

INFO: P [MainThread] 2025-09-15 17:10:34,700 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/predictor.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
approach_mapping = {
    "DO": "feature_perturbation",
    "def": "default_selftraining",
    "cfd": "confidence_threshold",
    "F1": "direct_eval",
    "torchem": "supervised_torchem",
    "_SF_": "supervised",
    "ST": "supervised_training",
}

In [16]:
models: Dict[str, List[str]] = {
    "HtoE": [
        "HmtoE_model4",
        "HmtoE_model_NA2",
        "HmtoE_model_Res1",
        # "HmtoE_model_Unetr2",
    ],
    "RtoE": [
        "RmtoE_model4",
        "RmtoE_model_NA2",
        "RmtoE_model_Res1",
        #"RmtoE_model_Unetr2",
    ],
    "VtoE": [
        "VtoE_model2",
        "VtoE_model_NA2",
        "VtoE_model_Res1",
    ],
    "EtoH": [
        "EtoHm_model5",
        "EtoHm_model_NA2",
        "EtoHm_model_Res1",
        #"EtoHm_model_Unetr2",
    ],
    "RtoH": [
        "RmtoHm_model4",
        "RmtoHm_model_NA2",
        "RmtoHm_model_Res1",
        #"RmtoHm_model_Unetr2",
    ],
    "VtoH": [
        "VtoHm_model2",
        "VtoHm_model_NA2",
        "VtoHm_model_Res1",
    ],
    "EtoR": [
        "EtoRm_model5",
        "EtoRm_model_NA2",
        "EtoRm_model_Res1",
        #"EtoRm_model_Unetr2",
    ],
    "HtoR": [
        "HmtoRm_model4",
        "HmtoRm_model_NA2",
        "HmtoRm_model_Res1",
        #"HmtoRm_model_Unetr2",
    ],
    "VtoR": [
        "VtoRm_model2",
        "VtoRm_model_NA2",
        "VtoRm_model_Res1",
    ],
    "EtoV": [
        "EtoV_model5",
        "EtoV_model_NA2",
        "EtoV_model_Res1",
        #"EtoV_model_Unetr2",
    ],
    "HtoV": [
        "HmtoV_model4",
        "HmtoV_model_NA2",
        "HmtoV_model_Res1",
        # "HmtoV_model_Unetr2",
    ],
    "RtoV": [
        "RmtoV_model4",
        "RmtoV_model_NA2",
        "RmtoV_model_Res1",
        #"RmtoV_model_Unetr2",
    ],
}

In [18]:
from typing import Union

def find_batchnorm_pred_path(
    model_name: str, 
    base_path: Union[Path, str], 
    perturbation: str = "none",
    return_summary: bool = False,
) -> Path:
    transfer = model_name.split("_")[0]
    source, target = transfer.split("to")
    pred_dir_path = Path(base_path) / f"{MODEL_ABBREVIATIONS_TO_DATASET[source]}_to_{MODEL_ABBREVIATIONS_TO_DATASET[target]}_gap" / model_name / "predictions" / perturbation / "predictions"
    if return_summary:
        pred_path = list(pred_dir_path.glob(f"metric_summary*.h5"))
    else:
        pred_path = list(pred_dir_path.glob("*predictions.h5"))
    assert len(pred_path) == 1, f"Found {len(pred_path)} prediction files for {model_name} at {pred_dir_path}"
    pred_path = pred_path[0]
    return pred_path

In [19]:
base_path = Path("/g/kreshuk/talks/model_ranking_results/AdaptiveBatchNorm/Mitochondria")
base_transfer_path = "/scratch/talks/consistency_results/patch_segmentation/mitochondria"
run_approach= "consistency"
run_id = "P_full"

In [21]:
#per_target_perf_results: Dict[str, Dict[str, Dict[str,float]]] = {}
per_target_perf_results: Dict[str, Dict[str, List[float]]] = {}
for transfer, model_list in models.items():
    print(f"Transfer: {transfer}")
    target = MODEL_ABBREVIATIONS_TO_DATASET[transfer[-1]]
    if target not in per_target_perf_results:
        per_target_perf_results[target] = {}
    transfer_gap = f"{MODEL_ABBREVIATIONS_TO_DATASET[transfer[0]]}_to_{target}_gap"
    for i, model_name in enumerate(model_list):
        pred_path = find_batchnorm_pred_path(model_name, base_path, perturbation="none", return_summary=True)

        median_eval_score_adjusted = load_h5(pred_path, "F1_eval_median")[1]

        yaml_paths = list((pred_path.parent.parent.parent.parent / "checkpoints").glob("*.yaml"))
        assert len(yaml_paths) == 1, f"Expected one YAML file for {model_name}, found {len(yaml_paths)}"
        yaml_path = yaml_paths[0]
        initial_model_name = Path(load_config_direct(yaml_path)[0]["model_cfg"]["source_checkpoint"]).parent.stem
        initial_pred_path = get_NA_prediction_path(
            model_name=initial_model_name,
            target=target,
            base_path=base_transfer_path,
            approach=run_approach,
            run_id=run_id,
        )
        eval_score_direct = load_h5(initial_pred_path, "hard_f1")
        median_eval_score = np.median(eval_score_direct, axis=0)[1]
        # per_target_perf_results[target].update({
        #     model_name : {"norm_Normalize": median_eval_score}
        # })
        per_target_perf_results[target].update({
            model_name : [median_eval_score, median_eval_score_adjusted]
        })


Transfer: HtoE
Transfer: RtoE
Transfer: VtoE
Transfer: EtoH
Transfer: RtoH
Transfer: VtoH
Transfer: EtoR
Transfer: HtoR
Transfer: VtoR
Transfer: EtoV
Transfer: HtoV
Transfer: RtoV


In [22]:
per_target_perf_results

{'EPFL': {'HmtoE_model4': [0.7714988, 0.0],
  'HmtoE_model_NA2': [0.6937871, 0.116159074],
  'HmtoE_model_Res1': [0.8338802, 0.13219377],
  'RmtoE_model4': [0.30063307, 0.0],
  'RmtoE_model_NA2': [0.10001396, 0.0],
  'RmtoE_model_Res1': [0.03238437, 0.116159074],
  'VtoE_model2': [0.14098334, 0.116159074],
  'VtoE_model_NA2': [0.60543156, 0.0],
  'VtoE_model_Res1': [0.47764683, 0.116159074]},
 'Hmito': {'EtoHm_model5': [0.3859456, 0.0],
  'EtoHm_model_NA2': [0.0, 0.2712749],
  'EtoHm_model_Res1': [0.21338218, 0.22706088],
  'RmtoHm_model4': [0.6027391, 0.0],
  'RmtoHm_model_NA2': [0.18514545, 0.0],
  'RmtoHm_model_Res1': [0.78607905, 0.13061824],
  'VtoHm_model2': [0.6169933, 0.57462794],
  'VtoHm_model_NA2': [0.44375935, 0.014595747],
  'VtoHm_model_Res1': [0.48414755, 0.28398675]},
 'Rmito': {'EtoRm_model5': [0.04665701, 0.0],
  'EtoRm_model_NA2': [0.0, 0.25217524],
  'EtoRm_model_Res1': [0.0, 0.25217524],
  'HmtoRm_model4': [0.8451842, 0.6559577],
  'HmtoRm_model_NA2': [0.8160182, 0